In [18]:
# %% 환경 설정
import os, re, time, json
from typing import List, Dict, Any
from dotenv import load_dotenv
from tqdm import tqdm
import jsonlines

# .env에서 OPENAI_API_KEY 읽기
# .env 예시:
# OPENAI_API_KEY=sk-...
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("OPENAI_API_KEY not found in environment. Check your .env file.")

# OpenAI 클라이언트 (Chat Completions 사용)
from openai import OpenAI
client = OpenAI(api_key=api_key)

# 파일 경로
IN_PATH  = "/Users/taeyoonkwack/Documents/PrivaCI-Bench/HF_cache/cases/GDPR/GDPR.jsonl"
OUT_UNIQUE_PATH = "0911_entities_unique.json"  # 전역 고유 엔티티 리스트

MODEL = "gpt-4o-mini"  # 빠르고 저렴 / 필요시 상향 가능
MAX_RETRIES = 5


In [19]:
# %% 프롬프트: "줄마다 1개 엔티티만" 강제
SYSTEM_PROMPT = (
    "You extract entities from short legal/regulatory text.\n"
    "- Output ONLY entities, one per line.\n"
    "- No JSON, no bullets, no numbers, no extra text.\n"
    "- If nothing, output an empty response."
)

def build_user_prompt(content: str) -> str:
    return (
        "Text:\n"
        f"{content}\n\n"
        "Return:\n"
        "- Entities ONLY, one per line.\n"
        "- No bullets, no numbering, no explanation.\n"
    )

def call_gpt_entities_lines(content: str, model: str = MODEL) -> str:
    """
    regulation_content에서 엔티티를 '줄바꿈으로만' 받은 원시 문자열로 반환.
    Chat Completions 사용. JSON 강제 안 함.
    """
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": build_user_prompt(content)}
                ],
                temperature=0.0,
                max_tokens=256,
            )
            return resp.choices[0].message.content or ""
        except Exception as e:
            if attempt == MAX_RETRIES:
                raise
            time.sleep(min(2 ** attempt, 10))
    return ""


In [20]:
# %% 라인 파싱 & 엔티티 정규화/중복 제거
_strip_leading = re.compile(r'^\s*(?:[-*•]+|\d+[\.\)]\s*)\s*')  # "- ", "* ", "1. " 등 제거
_whitespace_collapse = re.compile(r'\s+')

def parse_entities_from_lines(text: str) -> List[str]:
    """
    GPT가 반환한 멀티라인 텍스트를 엔티티 리스트로 변환.
    - 앞의 불릿/번호 제거
    - 공백 정리
    - 빈 라인 제거
    """
    out = []
    for line in (text or "").splitlines():
        s = _strip_leading.sub("", line)
        s = s.strip().strip('"\''"“”‘’`")  # 흔한 따옴표/백틱 제거
        s = _whitespace_collapse.sub(" ", s)
        if s:
            out.append(s)
    return out

def normalize_entity(e: str) -> str:
    e2 = e.lower().strip()
    e2 = _whitespace_collapse.sub(" ", e2)
    e2 = e2.strip('"\''"“”‘’()[]`")
    return e2

def dedupe_keep_order(items: List[str]) -> List[str]:
    seen = set()
    out = []
    for x in items:
        k = normalize_entity(x)
        if k not in seen and x:
            out.append(x)
            seen.add(k)
    return out


In [21]:
# %% 처리: 전역 고유 엔티티 리스트만 생성
global_unique_norm = set()
global_unique_list: List[str] = []

# input.jsonl 예시:
# {"reference":"{}", "regulation_id":"Article 1", "regulation_content":"\"Subject-matter and objectives\""}
# {"reference":"{}", "regulation_id":"Article 1(1)", "regulation_content":"\"This Regulation lays down ...\""}

with jsonlines.open(IN_PATH, mode="r") as reader:
    data = list(reader)  # tqdm를 위해 전체 길이 확보

for obj in tqdm(data, desc="Extracting unique entities", unit="line"):
    content = obj.get("case_content", "") or ""
    if not content.strip():
        continue

    # GPT 호출 → 줄바꿈 리스트
    raw = call_gpt_entities_lines(content)
    ents = parse_entities_from_lines(raw)
    ents = dedupe_keep_order(ents)  # 레코드 내부 중복 제거

    # 전역 고유 업데이트
    for ent in ents:
        key = normalize_entity(ent)
        if key not in global_unique_norm:
            global_unique_norm.add(key)
            global_unique_list.append(ent)

print(f"Processed lines: {len(data)}")
print(f"Global unique entities: {len(global_unique_list)}")


Extracting unique entities: 100%|██████████| 3137/3137 [1:11:36<00:00,  1.37s/line]

Processed lines: 3137
Global unique entities: 5701


In [22]:
# %% 저장
with open(OUT_UNIQUE_PATH, "w", encoding="utf-8") as f:
    json.dump(global_unique_list, f, ensure_ascii=False, indent=2)

print(f"Saved unique entities: {OUT_UNIQUE_PATH}")



Saved unique entities: 0911_entities_unique.json
